## 第 8 课：指针数学与 tl.trans

题目：[Triton: Matrix Transpose](https://www.deep-ml.com/problems/974?from=Triton%20Essentials)（ID 974）

计算目标：

In [ ]:
output[n, m] = x[m, n]

x 形状 `(M, N)`，output 形状 `(N, M)`，output 是连续 Tensor。

例如：

In [ ]:
x = [[1, 2, 3],
     [4, 5, 6]]

output = [[1, 4],
          [2, 5],
          [3, 6]]

这个 kernel **没有任何算术**，是纯粹的指针数学：难点是把一个 tile 的输入位置 `(m, n)` 翻译成输出位置 `(n, m)`。

### 1. 二维 tile 网格

每个 program 负责一个 `(BLOCK_M, BLOCK_N)` 的 tile：

In [ ]:
pid_m = tl.program_id(0)
pid_n = tl.program_id(1)
grid = (triton.cdiv(M, BLOCK_M), triton.cdiv(N, BLOCK_N))

### 2. 用广播构造二维指针矩阵

In [ ]:
offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)  # 形状 (BLOCK_M,)，作为“列向量”
offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)  # 形状 (BLOCK_N,)，作为“行向量”

x_ptrs = x_ptr + offs_m[:, None] * stride_xm + offs_n[None, :] * stride_xn  # (BLOCK_M, BLOCK_N)

`[:, None]` 和 `[None, :]` 广播后，`(i, j)` 位置就是 `x[offs_m[i], offs_n[j]]` 的地址。

二维 mask 用同样的广播：

In [ ]:
load_mask = (offs_m[:, None] < M) & (offs_n[None, :] < N)

### 3. tl.trans：在寄存器里换维度

加载的 tile 在寄存器里是 `(BLOCK_M, BLOCK_N)` 布局，但输出要求 `(n, m)` 排列：

In [ ]:
tile = tl.load(x_ptrs, mask=load_mask)
tile_t = tl.trans(tile)   # (BLOCK_M, BLOCK_N) → (BLOCK_N, BLOCK_M)

转置后第 0 维变成列下标 `offs_n`，第 1 维变成行下标 `offs_m`。

### 4. 输出侧的指针和 mask

输出是 `(N, M)` 的，tile 在输出里占 `(BLOCK_N, BLOCK_M)`：

In [ ]:
out_ptrs = out_ptr + offs_n[:, None] * stride_om + offs_m[None, :] * stride_on
store_mask = (offs_n[:, None] < N) & (offs_m[None, :] < M)

## 你的代码骨架

In [ ]:
import torch
import triton
import triton.language as tl


@triton.jit
def transpose_kernel(
    x_ptr,
    out_ptr,
    M,
    N,
    stride_xm,
    stride_xn,
    stride_om,
    stride_on,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
):
    # TODO 1：取得 tile 的行/列 program ID

    # TODO 2：生成 offs_m（BLOCK_M 个行下标，形状 (BLOCK_M,)）
    # 生成 offs_n（BLOCK_N 个列下标，形状 (BLOCK_N,)）

    # TODO 3：构造 (BLOCK_M, BLOCK_N) 的输入指针矩阵：
    # x_ptr + offs_m[:, None] * stride_xm + offs_n[None, :] * stride_xn

    # TODO 4：构造二维 load mask：(offs_m[:, None] < M) & (offs_n[None, :] < N)

    # TODO 5：加载 tile（形状 (BLOCK_M, BLOCK_N)）

    # TODO 6：tl.trans(tile) → 形状 (BLOCK_N, BLOCK_M)，第 0 维变成列下标

    # TODO 7：构造 (BLOCK_N, BLOCK_M) 的输出指针矩阵：
    # out_ptr + offs_n[:, None] * stride_om + offs_m[None, :] * stride_on

    # TODO 8：构造输出 mask 并 tl.store
    pass


def transpose(x: torch.Tensor) -> torch.Tensor:
    BLOCK_M = BLOCK_N = 32

    # TODO 9：分配 output（形状 (N, M)，与 x 同 dtype）

    # TODO 10：创建二维 grid：(cdiv(M, BLOCK_M), cdiv(N, BLOCK_N))

    # TODO 11：启动 kernel
    # stride 使用 x.stride(0), x.stride(1) 和 out.stride(0), out.stride(1)

    # TODO 12：返回 output
    pass

同时回答：

1. `M=64, N=48, BLOCK_M=BLOCK_N=32` 时，grid 是多少？边界 tile（`pid_m=1, pid_n=0`）里有多少个有效元素？
2. 为什么加载后要用 `tl.trans` 把 tile 换维度，而不是直接按 (m, n) 存、靠指针把顺序“转”过来？
3. 对连续矩阵，transpose 的读和写哪一侧是连续的？为什么这对性能很重要？

把代码和三个答案发给我，我继续审查。